In [1]:
!pip install -q diffusers transformers accelerate invisible-watermark>=0.2.0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image
from diffusers.utils import load_image, make_image_grid
import torch

RuntimeError: Failed to import diffusers.pipelines.auto_pipeline because of the following error (look up to see its traceback):
Failed to import diffusers.pipelines.controlnet.pipeline_controlnet_img2img because of the following error (look up to see its traceback):
No module named 'torch._prims_common'

In [ ]:
class StableDiffusionXl:
  def __init__(self):
    torch.cuda.empty_cache()
    self.pipeline_text2image = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
        ).to("cuda")
    self.pipeline_image2image = AutoPipelineForImage2Image.from_pipe(
        self.pipeline_text2image).to("cuda")

  def text2image(self, prompt):
    image = self.pipeline_text2image(prompt=prompt).images[0]
    return image

  def image2imageUrl(self, url, prompt, strength=0.8, guidance_scale=10.5):
    init_image = load_image(url)
    image = self.pipeline_image2image(prompt,
                                      image=init_image,
                                      strength=strength,
                                      guidance_scale=guidance_scale).images[0]
    return make_image_grid([init_image, image], rows=1, cols=2)


In [ ]:
stableDiffusionXl = StableDiffusionXl()

In [ ]:
image = stableDiffusionXl.text2image("Virat Kohli lifting the IPL tropy ")
image

In [ ]:
init_image = load_image("https://www.adorama.com/alc/wp-content/uploads/2018/11/landscape-photography-tips-yosemite-valley-feature.jpg")


In [ ]:
image = stableDiffusionXl.image2imageUrl("https://www.adorama.com/alc/wp-content/uploads/2018/11/landscape-photography-tips-yosemite-valley-feature.jpg",
                                      "Make this image anime style"
                                      )

In [ ]:
image

In [ ]:
image = stableDiffusionXl.text2image("Batman and robin")

In [ ]:
image


In [ ]:
pip install datasets


In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

In [ ]:
result = pipe("/content/1714934756247py9ock5h-voicemaker.in-speech.mp3") #Path to audio
print(result["text"])

In [ ]:
image = stableDiffusionXl.text2image(result["text"])
image